# 关系预测 —— WN18RR 词汇知识图谱

**任务定义**：给定头实体 $h$ 和尾实体 $t$，预测它们之间是哪种 WordNet 词汇关系。这是一个 11 类分类问题。

**数据集**：WN18RR 是从 WordNet 词汇数据库中抽取的知识图谱。其 11 种关系描述了英语单词（以 synset 编号表示）之间的语义和词汇联系：

| 关系 | 含义 | 示例 |
|------|------|------|
| `hypernym` | 上位词（is-a） | 狗 → 动物 |
| `instance_hypernym` | 实例-类别 | 北京 → 城市 |
| `derivationally_related_form` | 派生词 | run → runner |
| `has_part` | 整体-部分 | 汽车 → 轮胎 |
| `member_meronym` | 成员-群体 | 学生 → 班级 |
| `also_see` | 相关词 | 高兴 → 快乐 |
| `similar_to` | 近义形容词 | 大 → 巨大 |
| `verb_group` | 动词组 | 走 → 跑 |
| `member_of_domain_region` | 地域域 | 欧元 → 欧洲 |
| `member_of_domain_usage` | 用法域 | 医嘱 → 医学 |
| `synset_domain_topic_of` | 主题域 | 光合作用 → 植物学 |

**关键挑战**：`hypernym` 与 `instance_hypernym` 语义高度相似（都是“属于”关系），是主要的混淆来源。

**模型**：两阶段训练 —— 先用 TransE 链接预测预训练实体嵌入，再冻结 encoder 训练 RelationPredictionHead（MLP 分类器）。

In [ ]:
import sys
import time
import tempfile
import shutil
from collections import Counter
from pathlib import Path

import torch
import numpy as np
import matplotlib.pyplot as plt

_project_root = Path().resolve().parent
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from kge.config.loader import from_yaml
from kge.datasets import load_dataset, KGDataModule
from kge.models import build_model
from kge.trainer import create_trainer
from core.utils import get_device

from kge.notebooks import (
    plot_training_curves,
    plot_triple_split_summary,
    plot_confusion_matrix,
    format_relation_name,
)

%matplotlib inline
plt.rcParams["figure.dpi"] = 100

lp_cfg = from_yaml("config/link_prediction-baseline.yaml")
rp_cfg = from_yaml("config/relation_prediction.yaml")

print(f"Stage 1 (LP): {lp_cfg.model.encoder_name} on {lp_cfg.dataset.name}")
print(f"Stage 2 (RP): {rp_cfg.model.encoder_name} + {rp_cfg.model.head_name}")

## 数据探索

WN18RR 仅有 11 种关系，但 `_hypernym`（上位词）占据了约 40% 的三元组。关系分布高度不均衡，这对分类器是一个挑战。

In [ ]:
dataset = load_dataset("wn18rr", lp_cfg.dataset.root)
print(f"实体数量: {dataset.num_entities:,}")
print(f"关系数量: {dataset.num_relations:,}")
print(f"训练三元组: {len(dataset.train_triples):,}")
print(f"验证三元组: {len(dataset.val_triples):,}")
print(f"测试三元组: {len(dataset.test_triples):,}")

fig = plot_triple_split_summary(dataset)
plt.show()

id2rel = dataset.get_id_to_relation
rel_counter = Counter()
for triple in dataset.train_triples:
    rel_counter[int(triple[1])] += 1

wn18rr_rel_names = [
    format_relation_name(id2rel[i]) for i in range(dataset.num_relations)
]
print("\n各关系频率:")
total = sum(rel_counter.values())
for rid in sorted(rel_counter, key=rel_counter.get, reverse=True):
    name = wn18rr_rel_names[rid]
    count = rel_counter[rid]
    print(f"  {name:35s}: {count:>8,} ({count/total*100:5.1f}%)")

# 频率分布柱状图
fig2, ax = plt.subplots(figsize=(10, 4))
fig2.patch.set_facecolor("#ffffff")
names_sorted = [wn18rr_rel_names[rid] for rid, _ in rel_counter.most_common()]
counts_sorted = [c for _, c in rel_counter.most_common()]
bars = ax.bar(range(len(names_sorted)), counts_sorted, color="#3498db", alpha=0.8)
ax.set_xticks(range(len(names_sorted)))
ax.set_xticklabels(names_sorted, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("三元组数量")
ax.set_title("WN18RR 关系频率分布")
ax.grid(True, alpha=0.2, axis="y")
fig2.tight_layout()
plt.show()

## 阶段一：预训练实体嵌入（链接预测）

首先在 WN18RR 上训练 TransE 链接预测模型。虽然链接预测本身不是最终目标，但这一阶段为实体和关系学习到了有意义的低维嵌入，是下游关系预测的基础。

In [ ]:
device = get_device(lp_cfg.runtime.device)
print(f"设备: {device}")

data_module = KGDataModule(lp_cfg, dataset)
lp_model = build_model(lp_cfg, dataset.num_entities, dataset.num_relations)
print(f"Encoder 参数量: {sum(p.numel() for p in lp_model.encoder.parameters()):,}")

lp_trainer = create_trainer(lp_cfg, lp_model, data_module, device)

checkpoint_dir = Path(tempfile.mkdtemp(prefix="kge_rp_pretrain_"))
torch.manual_seed(42)
t0 = time.perf_counter()
lp_history = lp_trainer.train(checkpoint_dir)
print(f"预训练耗时: {time.perf_counter() - t0:.1f}s")

shutil.rmtree(checkpoint_dir, ignore_errors=True)

for k in sorted(lp_history):
    if k.startswith("test_"):
        print(f"LP Test {k.replace('test_', '').upper()}: {lp_history[k][0]:.4f}")

## 阶段二：关系预测模型

**RelationPredictionHead** 结构：
$$\text{MLP}([\mathbf{h} \oplus \mathbf{t}]) \in \mathbb{R}^{11}$$
- 输入：头实体和尾实体的嵌入拼接 $[\mathbf{h}; \mathbf{t}]$，维度 $2d$
- 隐层：Linear(2d, 256) → ReLU → Dropout(0.3)
- 输出：Linear(256, 11)，对应 11 种关系的 logits
- 训练时冻结 encoder，仅优化 head 参数
- 损失函数：CrossEntropy

In [ ]:
rp_model = build_model(rp_cfg, dataset.num_entities, dataset.num_relations)
rp_model.encoder.load_state_dict(lp_model.encoder.state_dict())

head_params = sum(p.numel() for p in rp_model.head.parameters())
print(f"Head 参数量: {head_params:,}")
print(f"Head 结构:\n{rp_model.head.mlp}")

rp_data_module = KGDataModule(rp_cfg, dataset)
rp_trainer = create_trainer(rp_cfg, rp_model, rp_data_module, device)

checkpoint_dir = Path(tempfile.mkdtemp(prefix="kge_rp_head_"))
torch.manual_seed(42)
t0 = time.perf_counter()
rp_history = rp_trainer.train(checkpoint_dir)
print(f"Head 训练耗时: {time.perf_counter() - t0:.1f}s")

shutil.rmtree(checkpoint_dir, ignore_errors=True)

for k in sorted(rp_history):
    if k.startswith("test_"):
        print(f"RP Test {k.replace('test_', '').upper()}: {rp_history[k][0]:.4f}")

## 训练曲线

关系预测训练器同时记录 train/val loss 和 accuracy，因此使用 `loss_mode="val_loss"` 模式，同时观察 loss 收敛和准确率提升。

In [ ]:
fig = plot_training_curves(
    rp_history, metric_key="acc", metric_label="Accuracy", loss_mode="val_loss"
)
plt.show()

test_acc = rp_history.get("test_acc", [0])[0]
print(f"Test Accuracy: {test_acc:.4f}")

## 评估：混淆矩阵

混淆矩阵是关系预测任务的核心诊断工具。重点关注：
- `hypernym` ↔ `instance_hypernym`：这两类关系的语义差异仅在于“普遍类别 vs 具体实例”，是模型最难区分的配对
- `derivationally_related_form` 是否与 `also_see` / `similar_to` 混淆

下面先使用 Trainer 的 `_eval` 逻辑手动评估测试集，再绘制行归一化混淆矩阵。

In [ ]:
@torch.no_grad()
def evaluate_relation_prediction(model, dataset, device):
    model.eval()
    triples = dataset.test_triples.to(device)
    h, r, t = triples[:, 0], triples[:, 1], triples[:, 2]

    bs = 1024
    all_preds, all_labels = [], []
    for i in range(0, len(triples), bs):
        batch_h = h[i:i+bs]
        batch_r = r[i:i+bs]
        batch_t = t[i:i+bs]
        logits = model(batch_h, batch_r, batch_t)
        all_preds.append(logits.argmax(dim=-1))
        all_labels.append(batch_r)

    return torch.cat(all_preds).cpu().numpy(), torch.cat(all_labels).cpu().numpy()

y_pred, y_true = evaluate_relation_prediction(rp_model, dataset, device)

fig = plot_confusion_matrix(
    y_true, y_pred, class_names=wn18rr_rel_names, normalize=True
)
plt.show()

from sklearn.metrics import classification_report
print(classification_report(y_true, y_pred, target_names=wn18rr_rel_names, digits=3))

## 总结

本 notebook 演示了在 WN18RR 上进行关系预测的完整两阶段流程：

1. **预训练**：TransE 链接预测为实体学习嵌入，在 WN18RR 上 MRR 约 0.2-0.3（11 种关系中 `hypernym` 占比高，预测相对容易）
2. **关系预测**：冻结 encoder 后训练 MLP 分类器，Head 仅 2d × 256 + 256 × 11 ≈ 50K 参数，训练速度快
3. **混淆分析**：
   - `hypernym` ↔ `instance_hypernym` 是主要混淆对，因为它们语义高度相似
   - 低频关系（如 `member_of_domain_region`）训练数据不足，召回率偏低
   - 派生词关系（`derivationally_related_form`）因其形态学特征明确，准确率较高

**局限与改进**：仅使用 `[h; t]` 拼接可能遗漏关系本身的细粒度语义信息；可考虑在 Head 中融入关系嵌入或使用注意力机制。